# Introduction to Groupby

## Notebook Outline
* [Introduction to Groupby](#IntroToGroupby)
* [Calculating Simple Statistics on Groups](#simplegroupstats)
* [Calculating Statistics on Multiple Columns in Groups, using .agg()](#agg)

In [ ]:
import pandas as pd
import numpy as np

# Introduction to Groupby

The groupby method allows to group data in a dataframe by one or more columns to create a 'groupby object'. We can then analyze each of the groups to create per-group-statistics and analysis.  For example, the mean temperature per month (this involves grouping by the month) or the total sales per shift manager (this involves grouping by the manager). Let's look at some examples!

<https://pandas.pydata.org/pandas-docs/stable/groupby.html>

In [ ]:
filepath = ('../Data/724080-13739-2001')
headers = ['Year', 'Month', 'Day', 'Hour', 'Air Temp', 'Dew Point Temp', 'Sea Level Pressure',
           'Wind Direction', 'Wind Speed Rate',
           'Sky Condition Total Coverage Code',
           'Liquid Precipitation Depth Dimension - 1Hr Duration',
           'Liquid Precipitation Depth Dimension - Six Hour Duration']
weatherData = pd.read_csv(filepath, delim_whitespace=True,
                          names=headers)
weatherData.loc[:, 'Air Temp'] = (weatherData['Air Temp']/10) * 1.8 + 32
weatherData.head(2)

#### Use .groupby() to group the data by the month

In [ ]:
districts_new2 = weatherData.groupby(['Year','Month','Day']).agg(['mean'])

In [ ]:
df_new=  weatherData.groupby(['Year','Month','Day']).apply(lambda weatherData: (weatherData['Air Temp']))

In [ ]:
df = weatherData[['Day']]*2

In [ ]:
print(df)

In [ ]:
print(df_new)

#### Let's explore our groupby object using the .groups attribute and the .size() method.
The .groups attribute will list the name of each group and the row index values that make up each group.

The .size() method will print the size of each group

In [ ]:
monthGroups.groups

In [ ]:
# it's not surprising that the February group has a smaller size since there
# are less days in February
monthGroups.size()

In [ ]:
monthGroups['Air Temp'].describe()

#### Getting groups using the .get_group() method
We can get a specific group by using the .get_group method. We just need to pass the name of the group which will be one of the values that we grouped by. See the example below

In [ ]:
# To get the March group we do the following:
marchData = monthGroups.get_group(3)
marchData.head(3)

# Calculating Simple Statistics on Groups

#### Now let's use the .mean(), .min(), and .max() methods on the groupby object to find the mean, min, and max air temperature of each month.

Note that operating on a groupby will return a DataFrame or Series depending on what the operation is.

In [ ]:
monthlyMeanTemps = monthGroups['Air Temp'].mean()
print(type(monthlyMeanTemps))
monthlyMeanTemps

In [ ]:
monthGroups['Air Temp'].max()

In [ ]:
monthGroups['Air Temp'].min()

#### We found a missing value! As a review, let's use some of what we have learned to take a closer look and fix it!

First let's use .loc[] and boolean series to find all values less than 10 (which is already 6 degrees lower than the minimum temp in January). The reason we do this is that I want to see if there are any other bad data points.

In [ ]:
weatherData.loc[weatherData['Air Temp'] < 10, :]

#### Let's replace it with the a linear interpolation between hour 7 and hour 9..
There are a few ways we can do this. For now, I am just going to use the .loc method to grab the rows corresponding to 2001-2-26 hours 7 and 9.

As a review, I will do this in a few steps:
* First I will use .loc to get the appropriate rows.
* Then, I will use .loc to get the rows for only the 'Air Temp' column
* Then, I will use the .mean() to get the mean value (same as a basic linear interpolation) all in one line.

In [ ]:
weatherData.loc[(weatherData['Month'] == 2) & (weatherData['Day'] == 26) &
                (weatherData['Hour'].isin([7, 9])), :]

In [ ]:
weatherData.loc[(weatherData['Month'] == 2) & (weatherData['Day'] == 26) &
                (weatherData['Hour'].isin([7, 9])), 'Air Temp']

In [ ]:
weatherData.loc[(weatherData['Month'] == 2) & (weatherData['Day'] == 26) &
                (weatherData['Hour'].isin([7, 9])), 'Air Temp'].mean()

#### Let's fill the bad value with this estimated value, using the .loc method.
Note that the row label of the row with the bad value is 1352. We will use that. Remember this is not the only way, this is just one way.

Note that the '\' allows me to continue the line of code on the next line. I will explain in lecture.

Also note that you must spell your columns names correctly - or you may accidentally add a new column!

In [ ]:
weatherData.loc[1352, 'Air Temp'] = \
    weatherData.loc[(weatherData['Month'] == 2) & (weatherData['Day'] == 26) &
                    (weatherData['Hour'].isin([7, 9])), 'Air Temp'].mean()

#### Finally, find the minimum temps of each month.

In [ ]:
monthGroups['Air Temp'].min()

# Using the .agg() method on a groupby objects
We can use the .agg() (short for aggregate) method on a group by object to calculate multiple statistics. Let's use the monthGroups object we have already created.

Docs are here: <https://pandas.pydata.org/pandas-docs/stable/generated/pandas.core.groupby.DataFrameGroupBy.agg.html>

In [ ]:
monthGroups['Air Temp'].agg(['min', 'mean', 'max', 'std'])

#### Other strings that pandas will recognize as functions are here:
<http://pandas.pydata.org/pandas-docs/stable/basics.html> (Scroll down about 25% to find the screenshot below)
![](functionnames.png)

#### Using .agg to compute different statistics on different columns:

In [ ]:
monthGroups.agg({'Air Temp': 'mean', 'Wind Speed Rate': 'max'})

#### Using .agg to compute different statistics on different columns, and renaming the columns:

In [ ]:
(monthGroups.agg({'Air Temp': 'mean', 'Wind Speed Rate': 'max'}).
 rename(columns={'Air Temp': 'Mean Air Temp', 'Wind Speed Rate': 'Max Wind Speed Rate'}))

# Using the apply method for custom analysis
The apply method allows us to apply a custom function to each group.

#### Finding the percentage of hourly temperature below 32
We can use the apply method to help us find the percentage of hourly temps that are below 32 in each month. First we need to define the function we will use.

In [ ]:
def prctTempsBelow32(x):
    # x is a series of hourly Air Temp values. All we need to do is test if
    # each value is less than 32. This will create a series of boolean values,
    # a value will be True if the temp is less than 32, and False if not.
    # Finding the mean of this boolean series gives us the percentage of hours
    # that are less than 32 (because True has a value of 1, and False a value
    # of 0)
    
    # The below code is great, but we could also do it all on one line:
    # return (x < 32).mean()
    
    tempLessThan32 = (x < 32).mean()
    return tempLessThan32

In [ ]:
monthGroups['Air Temp'].apply(prctTempsBelow32) * 100

#### What if we want the odds of any day in the month having at least one or more hourly temps below 32?

In [ ]:
def prctDayBelow32(x):
    # x is now a dataframe with rows only corresponding to a specific month
    # we will group this data by the day of the month
    dayGroups = x.groupby(by='Day')
    # now, we just need to test if the min temp on each day is below 32. If 
    # this is true, then we know that at least one hours is below 32
    daysLessThan32 = (dayGroups['Air Temp'].min() < 32)
    # daysLessThan32 is no a boolean series that is True if the days min temp
    # is less than 32, and false if not. Findinging the mean of this gives us
    # the percentages of days less than 32
    
    # This code works, but we could also do it all on one line:
    # return (x.groupby(by='Day')['Air Temp'].min() < 32).mean()
    return daysLessThan32.mean()

In [ ]:
monthGroups.apply(prctDayBelow32) * 100

#### Quick Review of Lambda Functions
A lambda function is a function that we define _in line_ that does not have a name. (In other programming languages these are sometimes called anonymous functions). For example,
instead of using the prctTempBelow32 function above we could just pass a lambda function to the apply method:

In [ ]:
monthGroups['Air Temp'].apply(lambda x: (x < 32).mean())

# Repeating The Same Analysis Using a Datetime Column

#### You can do the same on a datetime column, you just need to pass the datetime column itself to the groupby method:

First I am going to reload the data using the parse_dates argument to convert the first 4 columns to a datetime, and convert the 'Air Temp' column to Fahrenheit.

In [ ]:
weatherData = pd.read_csv(filepath, delim_whitespace=True,
                          names=headers, parse_dates=[[0, 1, 2, 3]])
weatherData.loc[:, 'Air Temp'] = (weatherData['Air Temp']/10) * 1.8 + 32
weatherData.head(2)

#### Below, note how I pass the datetime column, using .dt.month to get the month of each value in the column
So, really, I am passing a series of month values

In [ ]:
weatherData.groupby(by=weatherData['Year_Month_Day_Hour'].dt.month)['Air Temp'].mean()

## More Groupby Examples On Our Labor Sheet Data
The labor sheet is a great use case for groupby! Naturally, we may want to compare manager performance. Let's use groupy and some analysis to do!

First, we will load the dataset, just like we have before.

In [ ]:
filepath = ('../Data/LaborSheetData.csv')
laborSheetData = pd.read_csv(filepath, parse_dates=[[2, 3], 13])
laborSheetData.head()

#### Let's group the laborsheet data by the managers

In [ ]:
managerGroups = laborSheetData.groupby('Manager')
# let's print the size of the groups just to get an idea of what we are working with.
managerGroups.size()

#### Let's find the manager with the best mean DT TTL (drive through times)
To do this, we will use .mean() and then we will use .sort_values() to sort from the smallest DT TTL (fastest time) to the longest time.

In [ ]:
managerGroups['DT TTL'].mean().sort_values()

#### This is great, but we question Brittany S's time - I wonder how many hours she has worked?
Let's take into account the sample size by also finding the count of values for each manager. We need to use agg() for this. Aso, since the results have more than one column, we need to pass the name of the column that we want to sort by to the sort_values() method.

In [ ]:
managerDTTTL = (managerGroups['DT TTL'].agg(['mean', 'count']).
                sort_values('mean').rename(columns={'mean': 'Mean DT TTL',
                                                     'count': 'Sample Size'}))
print(managerDTTTL)

#### Let's filter out managers with a sample size less than 100.

In [ ]:
managerDTTTL.loc[managerDTTTL['Sample Size'] > 100, :].sort_values(by='Mean DT TTL')

#### Let's investigate the relationship between DT TTLs and the hour of the day.

In [ ]:
hourGroups = laborSheetData.groupby(laborSheetData['Date_Hour'].dt.hour)

In [ ]:
hourGroupDTTTLs = hourGroups['DT TTL'].agg(['mean', 'count'])
print(hourGroupDTTTLs)

#### What if we express manager DT TTLs in terms of deviations from the mean?
To do this we need to subtract the mean DT TTL for each of the recorded DT TTL values.
There are multiple ways to do this - we will show one way below.

#### Introducing the .to_dict() method to extract values from a dataframe to a dictionary.
Dictionaries are fast data structures - that are liberally used in python.

In [ ]:
meanHourDTTTLs = hourGroupDTTTLs['mean'].to_dict()
print(meanHourDTTTLs)

#### Now we loop through the dictionary and then subtract the mean from the appropriate rows in the dataframe:

In [ ]:
for hour in meanHourDTTTLs:
    laborSheetData.loc[laborSheetData['Date_Hour'].dt.hour == hour, 'DT TTL'] = \
    laborSheetData.loc[laborSheetData['Date_Hour'].dt.hour == hour, 'DT TTL'] - meanHourDTTTLs[hour]

In [ ]:
laborSheetData.head()

#### Now we use recalculate the mean DT TTLs, using the same groupby object!
Since we did not update anything in the managers column, we do not need to create a new groupby object!

In [ ]:
(managerGroups['DT TTL'].agg(['mean', 'count']).
                sort_values('mean').rename(columns={'mean': 'Mean DT TTL Deviations',
                                                     'count': 'Sample Size'}))

#### Let's now assign the output to a variable (a new dataframe) which we can then filter by the sample size

In [ ]:
managerDTTTLDev = (managerGroups['DT TTL'].agg(['mean', 'count']).
                sort_values('mean').rename(columns={'mean': 'Mean DT TTL Deviations',
                                                     'count': 'Sample Size'}))

In [ ]:
managerDTTTLDev.loc[managerDTTTLDev.loc[:, 'Sample Size'] > 100, :].sort_values(by='Mean DT TTL Deviations')

![](Success!.png)